# CSE 475 - Assignment 01
## Group Information

| Field | Details |
|-------------------------|-------------------------------------|
| **Group ID** | Group XX |
| **Student 1 Name** | [Your Full Name] |
| **Student 1 ID** | 20XX-X-XX-XXX |
| **Student 2 Name** | [Full Name] (if applicable) |
| **Student 2 ID** | 20XX-X-XX-XXX (if applicable) |
| **Notebook Type** | Transformer Models |
| **Dataset Source Name** | Tropical Flower Dataset |
| **Dataset Source Link** | https://www.kaggle.com/datasets/sabuktagin/tropical-flowers |
| **Kaggle Dataset Path** | /kaggle/input/tropical-flowers/ |
| **Submission Date** | DD March 2026 |

---

> **Note**: Please fill in your actual Group ID, Student Names, and IDs before submission.

# 🟣 Tropical Flower Classification — Vision Transformer Exploration

**Dataset**: Tropical Flower Dataset  
**Models**: ViT-B/16 · Swin-Tiny · DeiT-Small  

## 📋 Table of Contents
1. [⚙️ Global Configuration (START HERE)](#config)
2. [📦 Setup & Imports](#setup)
3. [📊 Exploratory Data Analysis](#eda)
4. [🔧 Data Preprocessing & Augmentation](#preprocessing)
5. [🏋️ Training Infrastructure](#training)
6. [🟣 Transformer Models: ViT · Swin · DeiT](#transformers)
7. [📈 Evaluation & Comparison](#evaluation)
8. [🛡️ Robustness Analysis](#robustness)
9. [🔍 Error Analysis](#error)



## ⚙️ Global Configuration ! <a id='config'></a>

**All hyperparameters and toggles are centralized here.**

In [13]:
import os
import re

# ╔══════════════════════════════════════════════════════════════════╗
# ║           GLOBAL CONFIGURATION — EDIT THIS CELL ONLY             ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Dataset ────────────────────────────────────────────────────────
DATA_DIR   = '/kaggle/input/datasets/sabuktagin/tropical-flowers/Tropical Flower Dataset Seven Species from Bangladesh for Classification and Ecological Research/Flower Dataset/Flower Dataset'
OUTPUT_DIR = '/kaggle/working'

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Safe filename helper (prevents issues like "ViT-B/16" creating subfolders)
def safe_name(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', str(name)).strip('_')

# ── Pretrained weights ─────────────────────────────────────────────
USE_PRETRAINED = False   # Change to True if you added weights dataset

# ── Image settings ─────────────────────────────────────────────────
IMG_SIZE    = 224    # Input resolution for transformers

# ── Training ───────────────────────────────────────────────────────
BATCH_SIZE      = 32     # Reduce to 16 if you get CUDA out-of-memory
NUM_EPOCHS      = 30     # Epochs per model (minimum 30 required per assignment)
ABLATION_EPOCHS = 1      # Epochs for ablation experiments
SEED            = 42
WARMUP_EPOCHS   = 3      # Warmup epochs for learning rate scheduler

# ── Optimizer ──────────────────────────────────────────────────────
CNN_LR          = 3e-4   # Learning rate for CNN models
TRANSFORMER_LR  = 1e-4   # Transformers need smaller LR
WEIGHT_DECAY    = 1e-4   # Weight decay for AdamW
GRAD_CLIP       = 1.0    # Max gradient norm for gradient clipping

# ── Mixed Precision Training ───────────────────────────────────────
USE_AMP         = True   # Enable Automatic Mixed Precision (torch.cuda.amp)

# ── Loss function ──────────────────────────────────────────────────
LABEL_SMOOTHING     = 0.1    # 0 = standard CE, 0.1 = slight smoothing
USE_CLASS_WEIGHTS   = True   # Weight loss by inverse class frequency

# ── Regularization ─────────────────────────────────────────────────
DROPOUT_RATE    = 0.3    # Dropout before classifier head
MIXUP_ALPHA     = 0.2    # MixUp augmentation strength (0 = disabled)
EARLY_STOP_PAT  = 6      # Patience for early stopping (epochs)

# ── Augmentation ───────────────────────────────────────────────────
RANDAUG_OPS     = 2      # RandAugment: number of operations
RANDAUG_MAG     = 9      # RandAugment: magnitude (1-30)
RANDOM_ERASE_P  = 0.1    # Probability of Random Erasing (Cutout)

# ── Model selection ────────────────────────────────────────────────
# Transformer timm model IDs
TRANSFORMER_MODELS = {
    'ViT-B/16':   'vit_base_patch16_224',
    'Swin-Tiny':  'swin_tiny_patch4_window7_224',
    'DeiT-Small': 'deit_small_patch16_224',
}

# ── Patch and Positional Embedding Configuration ───────────────────
PATCH_SIZES = {
    'ViT-B/16':   16,   # 224/16 = 14x14 = 196 patches
    'Swin-Tiny':  4,    # Initial patch size, then merged hierarchically
    'DeiT-Small': 16,   # Same as ViT
}
POSITIONAL_EMBEDDING_TYPES = {
    'ViT-B/16':   'learned',      # Learned 1D positional embeddings
    'Swin-Tiny':  'relative',     # Relative position bias
    'DeiT-Small': 'learned',      # Learned + distillation token
}

# ── GradCAM ────────────────────────────────────────────────────────
GRADCAM_SAMPLES_PER_CLASS = 2   # Images to visualize per class

# ── Robustness ─────────────────────────────────────────────────────
NOISE_LEVELS = [0.05, 0.15]     # Gaussian noise sigma values to test
BLUR_KERNELS = [3, 9]           # Blur kernel sizes to test
RESOLUTION_TESTS = [128, 192, 224, 256, 384]  # Resolutions for sensitivity test

# ── Display ────────────────────────────────────────────────────────
COLORS      = ['#E74C3C', '#F39C12', '#8E44AD', '#27AE60']
FIGDPI      = 120

# ── ImageNet normalization (DO NOT CHANGE) ──────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print("Configuration loaded successfully!")
print(f"   IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={NUM_EPOCHS}")
print(f"   USE_PRETRAINED={USE_PRETRAINED}, USE_AMP={USE_AMP}")
print(f"   Transformer LR={TRANSFORMER_LR}, Weight Decay={WEIGHT_DECAY}")
print(f"   Warmup epochs={WARMUP_EPOCHS}, Early stopping patience={EARLY_STOP_PAT}")
print(f"   Label smoothing={LABEL_SMOOTHING}, Class weighting={USE_CLASS_WEIGHTS}")
print("\nTransformer Models:")
for name, tid in TRANSFORMER_MODELS.items():
    patch = PATCH_SIZES.get(name, 'N/A')
    pos_emb = POSITIONAL_EMBEDDING_TYPES.get(name, 'N/A')
    print(f"   {name:<22} -> timm id: '{tid}' | patch={patch}, pos_emb={pos_emb}")

Configuration loaded successfully!
   IMG_SIZE=224, BATCH_SIZE=32, EPOCHS=30
   USE_PRETRAINED=False, USE_AMP=True
   Transformer LR=0.0001, Weight Decay=0.0001
   Warmup epochs=3, Early stopping patience=6
   Label smoothing=0.1, Class weighting=True

Transformer Models:
   ViT-B/16               -> timm id: 'vit_base_patch16_224' | patch=16, pos_emb=learned
   Swin-Tiny              -> timm id: 'swin_tiny_patch4_window7_224' | patch=4, pos_emb=relative
   DeiT-Small             -> timm id: 'deit_small_patch16_224' | patch=16, pos_emb=learned


In [14]:
# ── Dataset Classes ────────────────────────────────────────────────────────
NUM_CLASSES = 7
CLASS_NAMES = ['Bougainvillea', 'Crown of thorns', 'Hibiscus', 'Jungle geranium', 
               'Madagascar periwinkle', 'Marigold', 'Rose']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}

print('Classes:')
for i, c in enumerate(CLASS_NAMES):
    print(f'   [{i}] {c}')
print(f'Total: {NUM_CLASSES} classes')


Classes:
   [0] Bougainvillea
   [1] Crown of thorns
   [2] Hibiscus
   [3] Jungle geranium
   [4] Madagascar periwinkle
   [5] Marigold
   [6] Rose
Total: 7 classes


## 📦 Setup & Imports <a id='setup'></a>

In [15]:
# ── Install dependencies ────────────────────────────────────────────
# timm: PyTorch Image Models — ViT, Swin, DeiT, EfficientNet, etc.
# grad-cam: GradCAM / GradCAM++ for visualizing what the model sees
# tqdm: progress bars for training and evaluation loops
!pip install timm grad-cam tqdm -q


In [16]:
import os, random, time, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image

# tqdm
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn.functional as F

# TorchVision
from torchvision import datasets, transforms, models

# timm
import timm

# Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# GradCAM
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ── Reproducibility ────────────────────────────────────────────────
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\n📦 PyTorch {torch.__version__} | timm {timm.__version__}")

# ── Custom Dataset Class ──────────────────────────────────────────────
class FlowerDataset(Dataset):
    """Custom Dataset for Tropical Flower Classification."""
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


🖥️  Device : cuda
   GPU    : Tesla T4
   VRAM   : 15.6 GB

📦 PyTorch 2.10.0+cu128 | timm 1.0.25


NameError: name 'Dataset' is not defined

In [ ]:
# ── Paths & Data Loading ──────────────────────────────────────────────
from sklearn.model_selection import train_test_split

DATA_PATH = Path(DATA_DIR)
CLASS_NAMES = sorted([d.name for d in DATA_PATH.iterdir() if d.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}

print("📂 Classes:")
for i, c in enumerate(CLASS_NAMES):
    print(f"   [{i}] {c}")
print(f"\nTotal: {NUM_CLASSES} classes")

# ── Load all image paths ──────────────────────────────────────────────
def load_dataset_paths(data_dir):
    """Load all image paths and labels from directory structure."""
    data_path = Path(data_dir)
    image_paths = []
    labels = []
    
    for class_name in CLASS_NAMES:
        class_dir = data_path / class_name
        if class_dir.exists():
            for img_file in class_dir.iterdir():
                if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                    image_paths.append(str(img_file))
                    labels.append(CLASS_TO_IDX[class_name])
    
    return image_paths, labels

all_image_paths, all_labels = load_dataset_paths(DATA_DIR)
print(f"\n📊 Total images found: {len(all_image_paths)}")

# ── Train/Val/Test Split (80/10/10) ───────────────────────────────────
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_image_paths, all_labels, test_size=0.2, random_state=SEED, stratify=all_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, random_state=SEED, stratify=temp_labels
)

print(f"✅ Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")


## 📊 Exploratory Data Analysis <a id='eda'></a>

> **Why EDA first?**  
> We need to understand **class imbalance** (→ weighted loss), typical **image dimensions** (→ resize strategy),
> and **visual appearance** of each disease before choosing any model.

In [ ]:
def count_images_from_paths(paths, labels):
    """Count images per class from path/label lists."""
    counts = {c: 0 for c in CLASS_NAMES}
    for lbl in labels:
        counts[CLASS_NAMES[lbl]] += 1
    return counts

train_counts = count_images_from_paths(train_paths, train_labels)
test_counts  = count_images_from_paths(test_paths, test_labels)

print("Train set class distribution:")
for cls, cnt in train_counts.items():
    print(f"  {cls}: {cnt}")
print(f"\nTest set class distribution:")
for cls, cnt in test_counts.items():
    print(f"  {cls}: {cnt}")


In [ ]:
# ── Class distribution plots ───────────────────────────────────────
short = [c.replace(' Disease','').replace(' Leaf','') for c in CLASS_NAMES]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Tropical Flower Dataset — Class Distribution', fontsize=15, fontweight='bold')

# Grouped bar
x, w = np.arange(NUM_CLASSES), 0.35
axes[0].bar(x - w/2, stats_df['Train'], w, color=COLORS, alpha=0.85, label='Train')
axes[0].bar(x + w/2, stats_df['Test'],  w, color=COLORS, alpha=0.45, hatch='//', label='Test')
axes[0].set_xticks(x); axes[0].set_xticklabels(short, rotation=20, ha='right')
axes[0].set_title('Train vs Test'); axes[0].legend(); axes[0].set_ylabel('Count')
for b in axes[0].patches[:NUM_CLASSES]:
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+1,
                 str(int(b.get_height())), ha='center', fontsize=8)

# Pie
axes[1].pie(stats_df['Train'], labels=short, colors=COLORS,
            autopct='%1.1f%%', startangle=90, pctdistance=0.8)
axes[1].set_title('Training Distribution')

# Imbalance bar
ratio = stats_df['Train'] / stats_df['Train'].max()
bars = axes[2].barh(short, ratio, color=COLORS, alpha=0.85)
axes[2].axvline(1.0, color='k', ls='--', alpha=0.4, label='Majority class')
axes[2].set_xlabel('Relative proportion'); axes[2].set_title('Imbalance Ratio')
axes[2].set_xlim(0, 1.2); axes[2].legend()
for bar, v in zip(bars, ratio):
    axes[2].text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2,
                 f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sample images per class ────────────────────────────────────────
def sample_imgs(base, cls, n=5):
    files = [f for f in (base/cls).iterdir() if f.suffix.lower() in EXTS]
    return random.sample(files, min(n, len(files)))

N = 5
fig, axes = plt.subplots(NUM_CLASSES, N, figsize=(N*3, NUM_CLASSES*3))
fig.suptitle('Sample Images per Class (Training Set)', fontsize=14, fontweight='bold')

for r, cls in enumerate(CLASS_NAMES):
    for c, p in enumerate(sample_imgs(DATA_PATH, cls, N)):
        axes[r,c].imshow(Image.open(p).convert('RGB'))
        axes[r,c].axis('off')
        if c == 0:
            axes[r,c].set_ylabel(cls, fontsize=8, labelpad=48, rotation=0,
                                  ha='right', va='center', color=COLORS[r], fontweight='bold')
        if r == 0:
            axes[r,c].set_title(f'#{c+1}', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_images.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Image dimension & pixel statistics ────────────────────────────
print("⏳ Analyzing image properties (sampling 30 images/class)...")
records = []
for cls in CLASS_NAMES:
    for p in sample_imgs(DATA_PATH, cls, 30):
        img = Image.open(p).convert('RGB')
        arr = np.array(img)
        records.append({'class': cls, 'width': img.size[0], 'height': img.size[1],
                         'aspect': img.size[0]/img.size[1],
                         'mean_r': arr[:,:,0].mean(), 'mean_g': arr[:,:,1].mean(),
                         'mean_b': arr[:,:,2].mean(), 'std':    arr.std()})
props = pd.DataFrame(records)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Image Property Analysis', fontsize=13, fontweight='bold')

for i, cls in enumerate(CLASS_NAMES):
    sub = props[props['class']==cls]
    axes[0,0].hist(sub['width'],  alpha=0.6, color=COLORS[i], label=short[i], bins=15)
    axes[0,1].hist(sub['height'], alpha=0.6, color=COLORS[i], label=short[i], bins=15)
axes[0,0].set_title('Width Distribution');  axes[0,0].legend(fontsize=8)
axes[0,1].set_title('Height Distribution'); axes[0,1].legend(fontsize=8)

axes[0,2].boxplot([props[props['class']==c]['aspect'].values for c in CLASS_NAMES],
                   labels=short)
axes[0,2].set_title('Aspect Ratio'); axes[0,2].tick_params(axis='x', rotation=20)

props.groupby('class')[['mean_r','mean_g','mean_b']].mean().plot(
    kind='bar', ax=axes[1,0], color=['red','green','blue'], alpha=0.7)
axes[1,0].set_title('Mean RGB per Class'); axes[1,0].tick_params(axis='x', rotation=30)

for i, ch in enumerate(['mean_r','mean_g','mean_b']):
    axes[1,1].hist(props[ch], alpha=0.6, color=['red','green','blue'][i],
                   label=['R','G','B'][i], bins=20)
axes[1,1].set_title('Overall Channel Distribution'); axes[1,1].legend()

axes[1,2].boxplot([props[props['class']==c]['std'].values for c in CLASS_NAMES], labels=short)
axes[1,2].set_title('Pixel Std (Texture Complexity)'); axes[1,2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/image_properties.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()
print(f"Avg size: {props['width'].mean():.0f}×{props['height'].mean():.0f}px")

## 🔧 Data Preprocessing & Augmentation <a id='preprocessing'></a>

**Why ImageNet normalization?**  
Pretrained models were trained on ImageNet (μ=`[0.485,0.456,0.406]`, σ=`[0.229,0.224,0.225]`).  
Applying the same normalization ensures the input distribution matches what the backbone expects.

**Augmentation pipeline rationale:**

| Transform | Purpose |
|-----------|--------|
| `RandomCrop` | Scale invariance |
| `RandomHorizontalFlip` | Reflection symmetry |
| `ColorJitter` | Illumination invariance |
| `RandAugment` | Automatically learned policy for extra diversity |
| `RandomErasing` | Forces model to use full image, not single spot |

In [ ]:
# Class weights for imbalanced data handling
counts_arr = np.array([train_counts[c] for c in CLASS_NAMES], dtype=float)
class_weights = (1.0 / counts_arr) / (1.0 / counts_arr).sum() * NUM_CLASSES
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print(f'Class weights: {class_weights.cpu().numpy().round(2)}')


In [ ]:
# ── Visualize augmented batch ──────────────────────────────────────
def denorm(t):
    m = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    s = torch.tensor(IMAGENET_STD).view(3,1,1)
    return torch.clamp(t * s + m, 0, 1)

imgs_b, lbls_b = next(iter(train_loader))
fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle('Augmented Training Samples — same image can look very different each epoch!',
             fontsize=11, fontweight='bold')
for idx in range(min(32, BATCH_SIZE)):
    r, c = idx//8, idx%8
    axes[r,c].imshow(denorm(imgs_b[idx]).permute(1,2,0).numpy())
    axes[r,c].set_title(CLASS_NAMES[lbls_b[idx]].split()[0], fontsize=7)
    axes[r,c].axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/augmented_samples.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

## 🏋️ Training Infrastructure <a id='training'></a>

In [ ]:
# ── Training loop utilities with Warmup + AMP ────────────────────────────────

from torch.cuda.amp import autocast, GradScaler

def build_criterion():
    """Build loss function using CONFIG values."""
    return nn.CrossEntropyLoss(
        weight=CW_TENSOR if USE_CLASS_WEIGHTS else None,
        label_smoothing=LABEL_SMOOTHING
    )

def get_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    """
    Creates a warmup + cosine annealing learning rate scheduler.
    - Warmup: Linear increase from 0 to initial LR over warmup_epochs
    - Cosine: Cosine decay from initial LR to eta_min over remaining epochs
    """
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps = total_epochs * steps_per_epoch
    
    # Linear warmup scheduler
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer, 
        start_factor=0.01,  # Start at 1% of initial LR
        end_factor=1.0, 
        total_iters=warmup_steps
    )
    
    # Cosine annealing after warmup
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=total_steps - warmup_steps, 
        eta_min=1e-6
    )
    
    # Combine with SequentialLR
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_steps]
    )
    
    return scheduler

def train_epoch(model, loader, criterion, optimizer, scheduler=None, scaler=None, epoch=None, total_epochs=None):
    """
    Training epoch with optional mixed-precision (AMP) support.
    """
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    ep_label = f"Ep {epoch}/{total_epochs}" if epoch is not None else "Train"
    pbar = tqdm(loader, desc=ep_label, leave=False,
                bar_format="{l_bar}{bar:30}{r_bar}", colour="blue")
    
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        
        # Mixed-precision training with autocast
        if USE_AMP and scaler is not None:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        
        # Step scheduler per batch (for warmup + cosine)
        if scheduler:
            scheduler.step()
        
        loss_sum += loss.item()
        correct  += outputs.argmax(1).eq(labels).sum().item()
        total    += labels.size(0)
        pbar.set_postfix(loss=f"{loss_sum/len(loader):.4f}",
                         acc=f"{100.*correct/total:.2f}%")
    
    return loss_sum / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    """Returns (loss, accuracy, preds, labels, probs)."""
    model.eval()
    if criterion is None: criterion = nn.CrossEntropyLoss()
    loss_sum, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    pbar = tqdm(loader, desc="Eval", leave=False,
                bar_format="{l_bar}{bar:30}{r_bar}", colour="green")
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        out  = model(images)
        loss_sum += criterion(out, labels).item()
        probs     = F.softmax(out, dim=1)
        preds     = out.argmax(1)
        correct  += preds.eq(labels).sum().item()
        total    += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        pbar.set_postfix(acc=f"{100.*correct/total:.2f}%")
    return (loss_sum/len(loader), 100.*correct/total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))


def train_model(model, name, lr=TRANSFORMER_LR, epochs=NUM_EPOCHS):
    """
    Full training loop with:
    - AdamW optimizer with weight decay
    - Warmup + Cosine LR scheduling
    - Mixed-precision training (AMP)
    - Gradient clipping
    - Early stopping with checkpoint saving
    
    Returns (best_model, history_dict, best_val_accuracy)
    """
    model = model.to(DEVICE)
    criterion = build_criterion()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    
    # Warmup + Cosine LR Scheduler
    steps_per_epoch = len(train_loader)
    scheduler = get_warmup_cosine_scheduler(optimizer, WARMUP_EPOCHS, epochs, steps_per_epoch)
    
    # Mixed-precision scaler
    scaler = GradScaler() if USE_AMP else None

    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'lr':[]}
    best_acc, patience_ctr = 0.0, 0

    print(f"\n{'='*65}")
    print(f" Training: {name}")
    print(f" Optimizer: AdamW (lr={lr}, weight_decay={WEIGHT_DECAY})")
    print(f" Scheduler: Warmup ({WARMUP_EPOCHS} epochs) + Cosine Annealing")
    print(f" Mixed Precision (AMP): {USE_AMP}")
    print(f"{'='*65}")

    epoch_bar = tqdm(range(1, epochs + 1), desc=f"{name}",
                     bar_format="{l_bar}{bar:35}{r_bar}", colour="magenta")

    for ep in epoch_bar:
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer,
                                      scheduler, scaler, epoch=ep, total_epochs=epochs)
        va_loss, va_acc, _, _, _ = evaluate(model, test_loader, criterion)
        
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        history['lr'].append(current_lr)

        flag = " *" if va_acc > best_acc else ""
        epoch_bar.set_postfix(
            tr_loss=f"{tr_loss:.4f}", tr_acc=f"{tr_acc:.2f}%",
            va_loss=f"{va_loss:.4f}", va_acc=f"{va_acc:.2f}%{flag}",
            lr=f"{current_lr:.2e}",
            t=f"{time.time()-t0:.1f}s"
        )

        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f'{safe_name(name)}_best.pth'))
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= EARLY_STOP_PAT:
                tqdm.write(f"  Early stop at epoch {ep}")
                break

    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, f'{safe_name(name)}_best.pth')))
    print(f"\nBest val acc: {best_acc:.2f}%")
    return model, history, best_acc


def plot_history(history, name):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{name} - Training History', fontsize=12, fontweight='bold')
    ep = range(1, len(history['val_acc'])+1)
    
    # Loss plot
    axes[0].plot(ep, history['train_loss'], 'b-o', ms=3, label='Train')
    axes[0].plot(ep, history['val_loss'],   'r-o', ms=3, label='Val')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(alpha=.3)
    
    # Accuracy plot
    axes[1].plot(ep, history['train_acc'], 'b-o', ms=3, label='Train')
    axes[1].plot(ep, history['val_acc'],  'r-o', ms=3, label='Val')
    axes[1].set_title('Accuracy (%)')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(alpha=.3)
    
    # Learning rate plot
    if 'lr' in history:
        axes[2].plot(ep, history['lr'], 'g-o', ms=3)
        axes[2].set_title('Learning Rate (Warmup + Cosine)')
        axes[2].set_xlabel('Epoch')
        axes[2].set_yscale('log')
        axes[2].grid(alpha=.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{safe_name(name)}_history.png'), dpi=FIGDPI, bbox_inches='tight')
    plt.show()

print("Training utilities defined.")
print(f"  - AdamW optimizer with weight decay={WEIGHT_DECAY}")
print(f"  - Warmup ({WARMUP_EPOCHS} epochs) + Cosine Annealing LR")
print(f"  - Mixed Precision Training (AMP): {USE_AMP}")
print(f"  - Gradient clipping: max_norm={GRAD_CLIP}")

## 🟣 Transformer-Based Vision Models <a id='transformers'></a>

### How Vision Transformers Differ from CNNs

```
CNN path:   Image → Local Convolutions (local receptive field) → Hierarchy of features
ViT path:   Image → Patches → Linear Embed → Self-Attention (GLOBAL at every layer!)
```

| Property | CNN | Vision Transformer |
|----------|-----|-------------------|
| Receptive field | Local → grows with depth | Global from layer 1 |
| Inductive bias | Translation equivariance | Minimal (learned from data) |
| Data hunger | Low | High (ViT needs 14M+ images) |
| Scalability | Limited | Excellent (scales like LLMs) |

| Model | Year | Params | Key Innovation |
|-------|------|--------|----------------|
| **ViT-B/16** | 2020 | 86M | First pure-transformer image classifier (Dosovitskiy et al.) |
| **Swin-Tiny** | 2021 | 28M | Hierarchical + Shifted-Window attention → O(N) complexity |
| **DeiT-Small** | 2021 | 22M | Distillation token → train ViT with only ImageNet-1k |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Transformer Model 1: ViT-B/16                                   ║
# ╠══════════════════════════════════════════════════════════════════╣
# ║  Architecture:                                                    ║
# ║  - B/16: Base variant, 16×16 patches → 196 tokens for 224px     ║
# ║  - 12 transformer blocks, 768 hidden dim, 12 attention heads     ║
# ║  - [CLS] token aggregates representation for classification      ║
# ║  - Self-attention: Q×K^T/√d_k softmax × V  (Vaswani et al.)     ║
# ║  - Position embeddings: learned 1D sequence positions            ║
# ╚══════════════════════════════════════════════════════════════════╝

vit_b16 = timm.create_model(
    TRANSFORMER_MODELS['ViT-B/16'],
    pretrained=USE_PRETRAINED,
    num_classes=NUM_CLASSES,
    drop_rate=DROPOUT_RATE,
)

total_p = sum(p.numel() for p in vit_b16.parameters())
print(f"ViT-B/16  | Params: {total_p/1e6:.1f}M")
print(f"  Patch size : {vit_b16.patch_embed.patch_size}")
print(f"  Num patches: {vit_b16.patch_embed.num_patches}")
print(f"  Embed dim  : {vit_b16.embed_dim}")
print(f"  Num heads  : {vit_b16.blocks[0].attn.num_heads}")
print(f"  Depth      : {len(vit_b16.blocks)} blocks")

vit_b16, hist_vit, best_vit = train_model(vit_b16, 'ViT_B16', lr=TRANSFORMER_LR)
plot_history(hist_vit, 'ViT-B/16')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Transformer Model 2: Swin-Tiny                                  ║
# ╠══════════════════════════════════════════════════════════════════╣
# ║  Architecture:                                                    ║
# ║  - Hierarchical: 4 stages, resolution halved each stage          ║
# ║  - Window Attention: attend only within 7×7 local windows        ║
# ║  - Shifted Windows (SW-MSA): cross-window connections            ║
# ║    by shifting the window partition by (M/2, M/2)                ║
# ║  - Complexity: O(N·M²) vs ViT's O(N²) → much faster             ║
# ║  - Tiny: [2,2,6,2] blocks per stage, C=96 base channels         ║
# ╚══════════════════════════════════════════════════════════════════╝

swin_tiny = timm.create_model(
    TRANSFORMER_MODELS['Swin-Tiny'],
    pretrained=USE_PRETRAINED,
    num_classes=NUM_CLASSES,
    drop_rate=DROPOUT_RATE,
)

total_p = sum(p.numel() for p in swin_tiny.parameters())
print(f"Swin-Tiny | Params: {total_p/1e6:.1f}M")

swin_tiny, hist_swin, best_swin = train_model(swin_tiny, 'Swin_Tiny', lr=TRANSFORMER_LR)
plot_history(hist_swin, 'Swin-Tiny')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Transformer Model 3: DeiT-Small                                 ║
# ╠══════════════════════════════════════════════════════════════════╣
# ║  Architecture:                                                    ║
# ║  - Same structure as ViT but adds a [DISTILLATION] token         ║
# ║  - During training: [CLS] matches GT, [DIST] matches CNN teacher ║
# ║  - CNN teacher (RegNet) gives ViT inductive bias implicitly      ║
# ║  - At inference: average [CLS] + [DIST] softmax predictions      ║
# ║  - Small: 12 blocks, 384 dim, 6 heads                            ║
# ╚══════════════════════════════════════════════════════════════════╝

deit_small = timm.create_model(
    TRANSFORMER_MODELS['DeiT-Small'],
    pretrained=USE_PRETRAINED,
    num_classes=NUM_CLASSES,
    drop_rate=DROPOUT_RATE,
)

total_p = sum(p.numel() for p in deit_small.parameters())
print(f"DeiT-Small | Params: {total_p/1e6:.1f}M")

deit_small, hist_deit, best_deit = train_model(deit_small, 'DeiT_Small', lr=TRANSFORMER_LR)
plot_history(hist_deit, 'DeiT-Small')

## 📈 Evaluation & Comparison <a id='evaluation'></a>

In [ ]:
# ── Evaluate Transformer models ────────────────────────────────────
CNN_MODELS = {}  # no CNN models in this notebook

ALL_MODELS = {
    'ViT-B/16':   vit_b16,
    'Swin-Tiny':  swin_tiny,
    'DeiT-Small': deit_small,
}
ALL_HISTORIES = {
    'ViT-B/16':   hist_vit,
    'Swin-Tiny':  hist_swin,
    'DeiT-Small': hist_deit,
}

results = {}
crit_eval = nn.CrossEntropyLoss()

print(f"{'Model':<22} | {'Acc':>6} | {'F1':>6} | {'AUC':>6}")
print('-'*48)
for name, model in ALL_MODELS.items():
    _, acc, preds, labels, probs = evaluate(model, test_loader, crit_eval)
    lbin = label_binarize(labels, classes=range(NUM_CLASSES))
    auc_s = roc_auc_score(lbin, probs, multi_class='ovr', average='macro') * 100
    f1_s  = f1_score(labels, preds, average='macro') * 100
    results[name] = dict(accuracy=acc, f1=f1_s, auc=auc_s,
                          preds=preds, labels=labels, probs=probs)
    print(f"{name:<22} | {acc:>5.2f}% | {f1_s:>5.2f}% | {auc_s:>5.2f}%")

BEST_NAME = max(results, key=lambda k: results[k]['accuracy'])
print(f"\n🏆 Best Transformer: {BEST_NAME}  ({results[BEST_NAME]['accuracy']:.2f}%)")


In [ ]:
# ── Comparative bar charts ─────────────────────────────────────────
names = list(results.keys())
accs  = [results[m]['accuracy'] for m in names]
f1s   = [results[m]['f1']       for m in names]
aucs  = [results[m]['auc']      for m in names]
CNN_NAMES = list(CNN_MODELS.keys())
bcolors = ['#3498DB' if n in CNN_NAMES else '#E74C3C' for n in names]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
for ax, vals, title in zip(axes, [accs, f1s, aucs],
                            ['Test Accuracy (%)', 'Macro F1 (%)', 'Macro AUC (%)']):
    bars = ax.bar(names, vals, color=bcolors, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(max(0, min(vals)-10), 100)
    ax.tick_params(axis='x', rotation=35)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+.3,
                f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')
    ax.legend(handles=[
        mpatches.Patch(color='#3498DB', label='CNN'),
        mpatches.Patch(color='#E74C3C', label='Transformer')
    ], fontsize=8)
    ax.grid(axis='y', alpha=.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')

for ax, name in zip(axes.flat, names):
    cm  = confusion_matrix(results[name]['labels'], results[name]['preds'])
    cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cmn, annot=cm, fmt='d', ax=ax, cmap='Blues', vmin=0, vmax=1,
                xticklabels=short, yticklabels=short, cbar=False, linewidths=.5)
    ax.set_title(f'{name}  ({results[name]["accuracy"]:.1f}%)', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=25, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrices.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Classification report + per-class bar ─────────────────────────
br = results[BEST_NAME]
print(f"\n{'='*60}\nClassification Report — {BEST_NAME}\n{'='*60}")
print(classification_report(br['labels'], br['preds'], target_names=CLASS_NAMES, digits=4))

rep = classification_report(br['labels'], br['preds'],
                              target_names=CLASS_NAMES, output_dict=True)
fig, ax = plt.subplots(figsize=(12, 5))
x, w = np.arange(NUM_CLASSES), 0.25
ax.bar(x-w,   [rep[c]['precision'] for c in CLASS_NAMES], w, label='Precision', color='#3498DB', alpha=.8)
ax.bar(x,     [rep[c]['recall']    for c in CLASS_NAMES], w, label='Recall',    color='#E74C3C', alpha=.8)
ax.bar(x+w,   [rep[c]['f1-score']  for c in CLASS_NAMES], w, label='F1',        color='#2ECC71', alpha=.8)
ax.set_xticks(x); ax.set_xticklabels(short, rotation=15)
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=.3)
ax.set_title(f'Per-Class Metrics — {BEST_NAME}', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/per_class_metrics.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('ROC Curves — One-vs-Rest (OvR)', fontsize=14, fontweight='bold')

for ax, name in zip(axes.flat, names):
    lbin = label_binarize(results[name]['labels'], classes=range(NUM_CLASSES))
    probs = results[name]['probs']
    for ci, cn in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(lbin[:,ci], probs[:,ci])
        ax.plot(fpr, tpr, color=COLORS[ci], lw=2,
                label=f'{short[ci]} (AUC={auc(fpr,tpr):.2f})')
    ax.plot([0,1],[0,1],'k--',alpha=.4); ax.grid(alpha=.3)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'{name}  (AUC={results[name]["auc"]:.1f}%)', fontweight='bold')
    ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/roc_curves.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Convergence comparison ─────────────────────────────────────────
fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Training Convergence Comparison', fontsize=13, fontweight='bold')
styles = ['-','--',':','-.', '-','--']
pcolors = ['#3498DB','#1ABC9C','#2980B9','#E74C3C','#C0392B','#E67E22']

for (name, h), ls, pc in zip(ALL_HISTORIES.items(), styles, pcolors):
    ep = range(1, len(h['val_acc'])+1)
    a1.plot(ep, h['train_acc'], ls=ls, color=pc, alpha=.4)
    a1.plot(ep, h['val_acc'],   ls=ls, color=pc, lw=2, label=name)
    a2.plot(ep, h['val_loss'],  ls=ls, color=pc, lw=2, label=name)

a1.set_title('Val Accuracy (solid) / Train Accuracy (faded)')
a1.set_xlabel('Epoch'); a1.set_ylabel('Accuracy (%)')
a1.legend(fontsize=8); a1.grid(alpha=.3)
a2.set_title('Validation Loss'); a2.set_xlabel('Epoch')
a2.set_ylabel('Loss'); a2.legend(fontsize=8); a2.grid(alpha=.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/convergence.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

## 🛡️ Robustness Analysis <a id='robustness'></a>

### Why Robustness Matters

A Vision Transformer trained on clean, well-lit lab images may fail completely when deployed in the real world — on a farmer's phone with a dirty lens, in bright midday sun, or on compressed images shared over WhatsApp. **Clean-test accuracy is not enough.** Robustness analysis measures how gracefully a model degrades when inputs deviate from the training distribution.

---

### The Corruption Pipeline

We systematically inject six types of real-world degradation, each controlled by hyperparameters in the Global Config:

| Corruption | What it simulates | Controlled by |
|---|---|---|
| **Gaussian Noise σ** | Sensor noise, low-light photography | `NOISE_LEVELS` |
| **Gaussian Blur k** | Out-of-focus lens, motion blur | `BLUR_KERNELS` |
| **Overexposure** | Bright sunlight washing out colour | `ColorJitter brightness` |
| **Low Contrast** | Overcast sky, foggy conditions | `ColorJitter contrast` |
| **Rotation 45°** | Camera tilt, awkward hand angle | `RandomRotation` |

Each corruption is applied as a **deterministic transform** on the full test set — the model weights are frozen throughout; we are only changing the inputs.

---

### How Results Are Interpreted

For each corruption `c` we compute:
- **Accuracy(c)** — raw classification accuracy under that corruption
- **Drop(c) = Accuracy(clean) − Accuracy(c)** — percentage-point degradation

We use a three-tier colour coding:
- 🟢 **Drop < 5 pp** → Robust (acceptable for deployment)
- 🟡 **Drop 5–15 pp** → Moderately fragile (needs data augmentation)
- 🔴 **Drop > 15 pp** → Brittle (retraining with augmented data required)

> **Expected behaviour for ViT**: Transformers are generally more robust to global corruptions (noise, blur) than CNNs because self-attention aggregates *all* patch relationships rather than relying on local receptive fields. However, they can be more sensitive to geometric distortions because their position embeddings assume a fixed patch grid.

---

### GradCAM++ Explainability

After the robustness quantification, we run **GradCAM++** (Chattopadhyay et al., 2018) on clean test images to understand *where* the model is focusing. The heatmap is computed as:

```
heatmap = ReLU( Σ_k  α_k^c · A_k )
```

where `A_k` are the feature maps of the target layer and `α_k^c` are second-order gradient importance weights for class `c`. For ViT/DeiT we hook into the last transformer block's LayerNorm; for Swin we use the last stage's block norm.

Overlaying GradCAM++ on corrupted vs clean images reveals whether the model is looking at the **right region** (leaf surface, disease spots) or drifting to the background under corruption — a critical diagnostic for agricultural AI deployment.


In [ ]:
# ── Build corruption transforms ─────────────────────────────────────────────
# Each function returns a torchvision Compose pipeline that applies one specific
# corruption AFTER resizing to IMG_SIZE and BEFORE (or together with) normalisation.
# Keeping the ImageNet mean/std normalisation identical to val_transform ensures
# the corruption effect is isolated and not confounded by a different input scale.

def make_noise_transform(sigma):
    """
    Gaussian additive noise: x_corrupted = x + N(0, sigma^2)
    Applied in tensor space (post-ToTensor, so values in [0,1] before norm).
    sigma=0.05 → mild sensor noise; sigma=0.15 → heavy low-light noise.
    """
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.Lambda(lambda x: x + torch.randn_like(x) * sigma)
    ])

def make_blur_transform(k):
    """
    Gaussian blur with kernel size k (must be odd) and sigma = k/2.
    k=3 → subtle out-of-focus; k=9 → severe motion / lens blur.
    Applied in PIL space (pre-ToTensor) so it correctly blurs spatial structure.
    """
    k_odd = k if k % 2 == 1 else k + 1          # GaussianBlur requires odd kernel
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.GaussianBlur(kernel_size=k_odd, sigma=k / 2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])

# ── Assemble the corruption dictionary ───────────────────────────────────────
# Keys become chart labels; values are the transform pipelines.
CORRUPTIONS = {'✅ Clean (Baseline)': val_transform}

for s in NOISE_LEVELS:                           # from Config: e.g. [0.05, 0.15]
    CORRUPTIONS[f'Gaussian Noise σ={s}'] = make_noise_transform(s)

for k in BLUR_KERNELS:                           # from Config: e.g. [3, 9]
    CORRUPTIONS[f'Gaussian Blur k={k}'] = make_blur_transform(k)

CORRUPTIONS['Overexposure (bright)'] = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=(1.8, 1.8)),   # fixed brightness × 1.8
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

CORRUPTIONS['Low Contrast'] = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(contrast=(0.3, 0.3)),     # contrast reduced to 30 %
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

CORRUPTIONS['Rotation 45°'] = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=(45, 45)),     # deterministic 45° tilt
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# ── Run robustness evaluation ─────────────────────────────────────────────────
# For each corruption we build a fresh DataLoader from the TEST split so that
# the same images are used every time (only the transform changes).
# Model weights are NOT updated — we are testing, not training.
best_model = ALL_MODELS[BEST_NAME]
rob_results = {}

print(f"⏳ Robustness testing on: {BEST_NAME}")
print(f"   Baseline (clean) will be measured first for reference.\n")

rob_bar = tqdm(CORRUPTIONS.items(), desc="Corruptions", bar_format="{l_bar}{bar:30}{r_bar}")
for cname, ctransform in rob_bar:
    cds = FlowerDataset(test_paths, test_labels, transform=ctransform)
    cld = DataLoader(cds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    _, acc, preds, lbls, _ = evaluate(best_model, cld)
    f1_ = f1_score(lbls, preds, average='macro') * 100
    rob_results[cname] = {'accuracy': acc, 'f1': f1_}
    rob_bar.set_postfix(corruption=cname[:20], acc=f"{acc:.2f}%")

# ── Print summary table ───────────────────────────────────────────────────────
base_acc = rob_results['✅ Clean (Baseline)']['accuracy']
print(f"\n{'Corruption':<30} {'Acc':>7}  {'F1':>7}  {'Drop':>7}  {'Status'}")
print('-' * 65)
for name, res in rob_results.items():
    drop = base_acc - res['accuracy']
    status = ('🟢 Robust' if drop < 5 else ('🟡 Moderate' if drop < 15 else '🔴 Brittle'))
    print(f"{name:<30} {res['accuracy']:>6.2f}%  {res['f1']:>6.2f}%  {drop:>+6.2f}pp  {status}")


In [ ]:
# ── Robustness visualisation ──────────────────────────────────────────────────
# Two complementary charts:
#   LEFT  — Absolute accuracy per corruption (with baseline dashed line)
#   RIGHT — Accuracy *drop* (pp) relative to clean baseline
#            Colour bands mirror the 3-tier robustness rating above.

base_acc = rob_results['✅ Clean (Baseline)']['accuracy']
rnames = list(rob_results.keys())
raccs  = [rob_results[n]['accuracy'] for n in rnames]
drops  = [base_acc - a for a in raccs]          # positive = degradation

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Robustness Analysis — {BEST_NAME}', fontsize=13, fontweight='bold')

# ── Left: absolute accuracy ────────────────────────────────────────
rcolors = ['#27AE60' if n == '✅ Clean (Baseline)'
           else ('#E74C3C' if a < base_acc - 15 else '#F39C12')
           for n, a in zip(rnames, raccs)]
bars = ax1.bar(range(len(rnames)), raccs, color=rcolors, edgecolor='white', linewidth=0.5)
ax1.axhline(base_acc, color='green', ls='--', lw=2, alpha=.7,
            label=f'Baseline {base_acc:.1f}%')
ax1.set_xticks(range(len(rnames)))
ax1.set_xticklabels(rnames, rotation=40, ha='right', fontsize=8)
ax1.set_ylabel('Accuracy (%)'); ax1.set_title('Accuracy Under Corruptions')
ax1.legend(); ax1.set_ylim(0, 105); ax1.grid(axis='y', alpha=.3)
for b, a in zip(bars, raccs):
    ax1.text(b.get_x() + b.get_width() / 2, b.get_height() + .5,
             f'{a:.1f}', ha='center', fontsize=8, fontweight='bold')

# ── Right: degradation (drop) ──────────────────────────────────────
dcols = ['#27AE60' if d < 5 else ('#F39C12' if d < 15 else '#E74C3C') for d in drops]
ax2.barh(rnames, drops, color=dcols, edgecolor='white', linewidth=0.5)
ax2.axvline(0,  color='k',      lw=.8)
ax2.axvline(5,  color='orange', ls='--', alpha=.5, label='5 pp — robustness threshold')
ax2.axvline(15, color='red',    ls='--', alpha=.5, label='15 pp — fragility threshold')
ax2.set_xlabel('Accuracy Drop (pp)'); ax2.set_title('Degradation vs Baseline')
ax2.legend(fontsize=8); ax2.grid(axis='x', alpha=.3)
for i, d in enumerate(drops):
    ax2.text(d + .2, i, f'{d:.1f}pp', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/robustness.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()
print("✅ Robustness visualisation saved.")


In [ ]:
# ── GradCAM++ Explainability ───────────────────────────────────────
"""
GradCAM++ uses second-order gradients for better localization.
  heatmap = ReLU( Σ_k α_k^c · A_k )   where α are importance weights
Overlaying on the image shows WHICH pixels drive the prediction.
"""

def get_target_layer(model, name):
    if 'MobileNet' in name:
        return [model.conv_head]
    elif 'ResNeXt' in name:
        return [model.layer4[-1].conv3]
    elif 'EfficientNet' in name:
        return [model.conv_head]
    elif 'ViT' in name or 'DeiT' in name:
        return [model.blocks[-1].norm1]
    elif 'Swin' in name:
        return [model.layers[-1].blocks[-1].norm1]
    return [list(model.children())[-2]]

def raw_and_tensor(img_path):
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    raw = np.array(img) / 255.0
    t   = val_transform(img).unsqueeze(0).to(DEVICE)
    return raw, t

try:
    cam = GradCAMPlusPlus(model=best_model, target_layers=get_target_layer(best_model, BEST_NAME))
    n   = GRADCAM_SAMPLES_PER_CLASS
    fig, axes = plt.subplots(NUM_CLASSES*n, 3, figsize=(10, NUM_CLASSES*n*3))
    fig.suptitle(f'GradCAM++ — {BEST_NAME}  (Original | Map | Overlay)', fontsize=12, fontweight='bold')

    row = 0
    for ci, cls in enumerate(CLASS_NAMES):
        for p in sample_imgs(DATA_PATH, cls, n):
            raw, t = raw_and_tensor(p)
            with torch.no_grad():
                out  = best_model(t)
                pred = out.argmax(1).item()
                conf = F.softmax(out,1)[0,pred].item()
            gcam = cam(t, [ClassifierOutputTarget(pred)])[0]
            over = show_cam_on_image(raw.astype(np.float32), gcam, use_rgb=True)

            tick = '✅' if pred == ci else '❌'
            axes[row,0].imshow(raw);     axes[row,0].set_title(f'True: {short[ci]}', fontsize=8)
            axes[row,1].imshow(gcam, cmap='jet'); axes[row,1].set_title('GradCAM++ Map', fontsize=8)
            axes[row,2].imshow(over);
            axes[row,2].set_title(f'{tick} {short[pred]} ({conf:.0%})', fontsize=8)
            for ax in axes[row]: ax.axis('off')
            row += 1

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/gradcam.png', dpi=FIGDPI, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f"GradCAM skipped: {e}")

## 🔍 Error Analysis <a id='error'></a>

### Why Error Analysis Goes Beyond Accuracy

A single accuracy number hides *where* and *why* a model fails. Error analysis disaggregates failures across three dimensions:

1. **Confidence** — Is the model well-calibrated, or does it make wrong predictions with high confidence?
2. **Per-class patterns** — Are certain disease categories systematically harder to recognise?
3. **Confusion structure** — Which class pairs are most easily mixed up, and does visual similarity explain it?

---

### Process — Step by Step

**Step 1 — Confidence Distribution (`Cell 33`, top-left)**

We plot two histograms of the model's maximum softmax probability:
- 🟢 **Correct predictions** — ideally peaked near 1.0 (confident and right)
- 🔴 **Wrong predictions** — ideally peaked near 0.5 (uncertain and wrong)

If wrong predictions also peak near 1.0, the model is **overconfident** — a dangerous property in medical/agricultural AI where a confident wrong prediction is worse than an uncertain one.

**Step 2 — Reliability Diagram / Calibration Curve (`Cell 33`, top-right)**

We bin predictions by confidence (10 bins from 0→1) and measure the true accuracy in each bin. A **perfectly calibrated** model lies on the diagonal: "when I say 80% confident, I am right 80% of the time." Deviation above the diagonal = underconfident; below = overconfident.

> ViT models trained with label smoothing tend to be better calibrated than those without, because soft targets prevent the model from pushing logits to ±∞.

**Step 3 — Per-Class Error Rate (`Cell 33`, bottom-left)**

Error rate = 1 − per-class accuracy. High error on a specific class suggests:
- **Insufficient training samples** for that class (→ check class balance from EDA)
- **High intra-class visual variability** (early vs late disease stage look different)
- **High inter-class visual similarity** (two diseases with similar colouration)

**Step 4 — Top Confusion Pairs (`Cell 33`, bottom-right)**

The off-diagonal entries of the confusion matrix, sorted by count. The most frequent confusion pair reveals which two classes the model treats as nearly indistinguishable — a direct signal for where to focus data collection or specialised augmentation.

**Step 5 — High-Confidence Wrong Predictions (`Cell 34`)**

We filter all test images where: `prediction ≠ ground truth AND confidence > 70%`. These are the **hardest errors** — the model was not just wrong, it was *sure* it was right. Visualising these images often reveals:
- Ambiguous or mislabelled ground-truth samples in the dataset
- Edge-case disease presentations the training set never covered
- Systematic background / lighting biases the model has latched onto (cross-check with GradCAM++)

---

### Interpreting Results

| Finding | Likely cause | Recommended action |
|---|---|---|
| Wrong preds cluster at high confidence | Overconfident model | Add label smoothing; use temperature scaling |
| Calibration curve bows below diagonal | Overconfident | Increase dropout; increase label smoothing |
| One class has >> 30% error rate | Class imbalance or ambiguity | Oversample + targeted augmentation |
| Two classes dominate confusion pairs | Visual similarity | Collect more discriminative examples |
| High-conf errors are visually reasonable | Dataset label noise | Manual review + re-label |


In [ ]:
# ── Confidence & Calibration Analysis ────────────────────────────────────────
# We pull the stored prediction arrays from the `results` dict populated during
# the evaluation phase (Cell 22).  No model forward pass is needed here — all
# computations work on the cached preds/labels/probs arrays.

preds_  = results[BEST_NAME]['preds']    # shape (N,)   — argmax class index
labels_ = results[BEST_NAME]['labels']   # shape (N,)   — ground-truth class index
probs_  = results[BEST_NAME]['probs']    # shape (N, C) — softmax probabilities
maxp    = probs_.max(axis=1)             # shape (N,)   — max prob = model confidence
correct = (preds_ == labels_)            # shape (N,)   — boolean mask

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Error Analysis — {BEST_NAME}', fontsize=13, fontweight='bold')

# ── Plot 1: Confidence histogram (correct vs wrong) ───────────────────────────
# Ideal: correct predictions → confidence near 1.0
#        wrong predictions   → confidence near 0.5 (model was uncertain)
# Bad:   wrong predictions also at high confidence → overconfident / miscalibrated
axes[0, 0].hist(maxp[correct],  bins=25, alpha=.7, color='green',
                label=f'Correct  (n={correct.sum()})',   density=True)
axes[0, 0].hist(maxp[~correct], bins=25, alpha=.7, color='red',
                label=f'Wrong    (n={(~correct).sum()})', density=True)
axes[0, 0].set_xlabel('Confidence (max softmax prob)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Confidence Distribution: Correct vs Wrong')
axes[0, 0].legend()
axes[0, 0].axvline(.5, color='gray', ls='--', alpha=.5, label='0.5 threshold')

# ── Plot 2: Reliability Diagram (Calibration Curve) ───────────────────────────
# Bins predictions into 10 confidence intervals [0.0,0.1), [0.1,0.2), …, [0.9,1.0]
# For each bin we measure the fraction of predictions that were actually correct.
# Perfect calibration → all bars at the diagonal y = x.
bins  = np.linspace(0, 1, 11)
bc    = (bins[:-1] + bins[1:]) / 2    # bin centres for plotting
cal_acc, bin_sizes = [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (maxp >= lo) & (maxp < hi)
    cal_acc.append(correct[mask].mean() if mask.sum() > 0 else 0)
    bin_sizes.append(mask.sum())

axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=.5, label='Perfect calibration')
bars_cal = axes[0, 1].bar(bc, cal_acc, width=.09, alpha=.7,
                           color='steelblue', label='Model')
axes[0, 1].set_title('Reliability Diagram (Calibration Curve)')
axes[0, 1].set_xlabel('Confidence'); axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend(); axes[0, 1].set_xlim(0, 1); axes[0, 1].set_ylim(0, 1)
# Annotate each bar with the number of samples in that bin
for b, n_b in zip(bars_cal, bin_sizes):
    if n_b > 0:
        axes[0, 1].text(b.get_x() + b.get_width() / 2, b.get_height() + .02,
                        str(n_b), ha='center', fontsize=7, color='gray')

# ── Plot 3: Per-class Error Rate ───────────────────────────────────────────────
# Error rate = 1 − (correct predictions for class i / total samples of class i)
# A class with > 30% error is a red flag and warrants investigation.
cls_errs = [
    1 - correct[labels_ == i].mean() if (labels_ == i).sum() > 0 else 0
    for i in range(NUM_CLASSES)
]
bars_cls = axes[1, 0].bar(short, [e * 100 for e in cls_errs], color=COLORS,
                           edgecolor='white', linewidth=0.5)
axes[1, 0].axhline(100 * (1 - correct.mean()), color='gray', ls='--', alpha=.6,
                    label=f'Overall error {100*(1-correct.mean()):.1f}%')
axes[1, 0].set_ylabel('Error Rate (%)'); axes[1, 0].set_title('Per-Class Error Rate')
axes[1, 0].tick_params(axis='x', rotation=20); axes[1, 0].grid(axis='y', alpha=.3)
axes[1, 0].legend(fontsize=8)
for b, e in zip(bars_cls, cls_errs):
    axes[1, 0].text(b.get_x() + b.get_width() / 2, b.get_height() + .3,
                    f'{e * 100:.1f}%', ha='center', fontsize=9, fontweight='bold')

# ── Plot 4: Top Confusion Pairs (off-diagonal CM entries) ─────────────────────
# Each off-diagonal entry cm[i,j] means "true class i was predicted as class j".
# Sorting by count surfaces the most frequent systematic errors.
cm = confusion_matrix(labels_, preds_)
pairs = [
    (CLASS_NAMES[i].split()[0] + ' → ' + CLASS_NAMES[j].split()[0], cm[i, j])
    for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)
    if i != j and cm[i, j] > 0
]
pairs.sort(key=lambda x: -x[1])
if pairs:
    pair_names, pair_counts = zip(*pairs[:8])
    axes[1, 1].barh(pair_names, pair_counts, color='#E74C3C', alpha=.8,
                    edgecolor='white', linewidth=0.5)
    axes[1, 1].set_xlabel('Count'); axes[1, 1].set_title('Top-8 Confusion Pairs')
    axes[1, 1].grid(axis='x', alpha=.3)
    for i, cnt in enumerate(pair_counts):
        axes[1, 1].text(cnt + .2, i, str(cnt), va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/error_analysis.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

# ── Print ECE (Expected Calibration Error) ────────────────────────────────────
# ECE = weighted average of |accuracy − confidence| across bins
ece = sum(abs(acc_b - conf_b) * n_b
          for acc_b, conf_b, n_b in zip(cal_acc, bc, bin_sizes)) / len(labels_)
print(f"\n📊 Expected Calibration Error (ECE) = {ece:.4f}")
print(f"   (0.00 = perfect; < 0.05 = well-calibrated; > 0.10 = overconfident)")
print(f"   Overall Accuracy : {correct.mean()*100:.2f}%")
print(f"   Mean Confidence  : {maxp.mean()*100:.2f}%")
print(f"   Gap (conf−acc)   : {(maxp.mean() - correct.mean())*100:+.2f}pp")


In [ ]:
# ── High-Confidence Wrong Predictions ────────────────────────────────────────
# These are the most dangerous errors: the model is not merely wrong, it is
# *confidently* wrong (confidence > 70%).  Visualising these images helps
# distinguish three root causes:
#   (a) Genuinely ambiguous images / mislabelled ground truth
#   (b) OOD (out-of-distribution) samples the training set never covered
#   (c) Spurious correlations (model attending to background, not the leaf)
# Cross-checking with GradCAM++ above is strongly recommended.

test_no_aug = FlowerDataset(test_paths, test_labels, transform=val_transform)
test_ldr1   = DataLoader(test_no_aug, batch_size=1, shuffle=False)

hard_errors  = []
CONF_THRESHOLD = 0.70   # errors above this confidence are considered "high-confidence"

print(f"Scanning test set for high-confidence errors (conf > {CONF_THRESHOLD:.0%})...")
with torch.no_grad():
    scan_bar = tqdm(enumerate(test_ldr1), total=len(test_ldr1),
                    desc="Scanning", bar_format="{l_bar}{bar:30}{r_bar}", leave=False)
    for i, (img, lbl) in scan_bar:
        out  = best_model(img.to(DEVICE))
        prob = F.softmax(out, 1)[0]
        pred = out.argmax(1).item()
        conf = prob[pred].item()
        if pred != lbl.item() and conf > CONF_THRESHOLD:
            hard_errors.append({
                'conf': conf,
                'true': lbl.item(),
                'pred': pred,
                'img':  img[0].clone()
            })

hard_errors.sort(key=lambda x: -x['conf'])   # worst first (highest conf, still wrong)
print(f"\n⚠️  High-confidence errors found: {len(hard_errors)}")
print(f"   These represent {len(hard_errors)/len(test_no_aug)*100:.2f}% of the test set.\n")

# Summarise by true class
print("Breakdown by true class:")
from collections import Counter
tc = Counter(e['true'] for e in hard_errors)
for cls_idx, cnt in sorted(tc.items(), key=lambda x: -x[1]):
    print(f"  {CLASS_NAMES[cls_idx]:<35} → {cnt} high-conf errors")

# ── Visualise top errors ──────────────────────────────────────────────────────
n_show = min(12, len(hard_errors))
if n_show > 0:
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle(
        f'Top-{n_show} High-Confidence Errors — {BEST_NAME}\n'
        f'(Sorted by descending confidence — most dangerous mistakes first)',
        fontsize=11, fontweight='bold'
    )
    for idx, err in enumerate(hard_errors[:n_show]):
        r, c = idx // 4, idx % 4
        img_np = denorm(err['img']).permute(1, 2, 0).numpy()
        axes[r, c].imshow(img_np)
        axes[r, c].set_title(
            f"True : {short[err['true']]}\n"
            f"Pred : {short[err['pred']]}\n"
            f"Conf : {err['conf']:.1%}",
            fontsize=8, color='darkred'
        )
        axes[r, c].axis('off')
        # Red border around each image to draw attention
        for spine in axes[r, c].spines.values():
            spine.set_edgecolor('red'); spine.set_linewidth(2)

    for idx in range(n_show, 12):        # hide unused subplots
        axes[idx // 4, idx % 4].axis('off')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/high_conf_errors.png', dpi=FIGDPI, bbox_inches='tight')
    plt.show()
    print("\n💡 Next step: cross-reference these images with GradCAM++ to check")
    print("   whether the model is attending to the leaf or the background.")
else:
    print("✅ No high-confidence errors found — excellent calibration!")


In [ ]:
# ── Final summary table ────────────────────────────────────────────
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

sum_rows = []
for name, m in ALL_MODELS.items():
    p = sum(x.numel() for x in m.parameters())
    sum_rows.append({
        'Model': name,
        'Type': 'CNN' if name in CNN_MODELS else 'Transformer',
        'Params': f'{p/1e6:.1f}M',
        'Accuracy': f"{results[name]['accuracy']:.2f}%",
        'F1 Macro': f"{results[name]['f1']:.2f}%",
        'AUC':      f"{results[name]['auc']:.2f}%",
    })
print(pd.DataFrame(sum_rows).to_string(index=False))
print(f"\n🏆 Best model: {BEST_NAME} ({results[BEST_NAME]['accuracy']:.2f}%)")

print("""
╔══════════════════════════════════════════════════════════════════╗
║  KEY TAKEAWAYS FOR STUDENTS                                       ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. Transfer learning >> Random init, especially with small data  ║
║  2. Augmentation is your free lunch — use it aggressively         ║
║  3. CNNs still competitive (especially MobileNet) for small sets  ║
║  4. Transformers shine when data > ~100k images (or pretrained)   ║
║  5. Label smoothing: prevents overconfident/miscalibrated models  ║
║  6. GradCAM is essential — always check WHERE the model looks     ║
║  7. Robustness matters more than clean-test accuracy in practice  ║
╚══════════════════════════════════════════════════════════════════╝
""")

---

## 📚 References

1. **MobileNetV3**: Howard et al. (2019). *Searching for MobileNetV3*. ICCV 2019.
2. **ResNeXt**: Xie et al. (2017). *Aggregated Residual Transformations for Deep Neural Networks*. CVPR 2017.
3. **EfficientNet**: Tan & Le (2019). *EfficientNet: Rethinking Model Scaling for CNNs*. ICML 2019.
4. **ViT**: Dosovitskiy et al. (2020). *An Image is Worth 16x16 Words*. ICLR 2021.
5. **Swin**: Liu et al. (2021). *Swin Transformer: Hierarchical ViT using Shifted Windows*. ICCV 2021.
6. **DeiT**: Touvron et al. (2021). *Training data-efficient image transformers*. ICML 2021.
7. **GradCAM++**: Chattopadhyay et al. (2018). *Grad-CAM++*. WACV 2018.
8. **Label Smoothing**: Müller et al. (2019). *When Does Label Smoothing Help?*. NeurIPS 2019.
9. **AdamW**: Loshchilov & Hutter (2019). *Decoupled Weight Decay Regularization*. ICLR 2019.
10. **RandAugment**: Cubuk et al. (2020). *RandAugment: Practical automated data augmentation*. CVPR 2020.

---
*East West University — CSE 445 / CSE 475*